In [ ]:
import pandas as pd

df = pd.read_csv("/mnt/d/Main/Core/DataSets/DL/cx/100_Unique_QA_Dataset.csv")

df.head()

,question,answer
0,What is Hybridizer?,Hybridizer is a compiler from Altimesh that en...
1,How does Hybridizer generate optimized code?,Hybridizer uses decorated symbols to express p...
2,What are some parallelization patterns mention...,The text mentions using parallelization patter...
3,How can you benefit from accelerators without ...,You can benefit from accelerators' compute hor...
4,What is an example of using Hybridizer?,An example in the text demonstrates using Para...


In [29]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [30]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [31]:
# vocab
vocab = {'<UNK>':0}

In [32]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)


In [33]:
df.apply(build_vocab, axis=1)

0       None
1       None
2       None
3       None
4       None
        ... 
7103    None
7104    None
7105    None
7106    None
7107    None
Length: 7108, dtype: object

In [34]:
len(vocab)

12240

In [35]:
# convert words to numerical indices
def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [36]:
text_to_indices("What is campusx", vocab)

[1, 2, 0]

In [37]:
import torch
from torch.utils.data import Dataset, DataLoader

In [38]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [39]:
dataset = QADataset(df, vocab)

In [40]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [41]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[    1,     2, 11618,    12,    20,     2,   104,   216,    42,  1177,
           315,  3262]]) tensor([11619,     2,     4,   186,   216,    27,   944,   315,  3262,     6,
         5334,  4630,   500,   501,  3336,   524,   135,  3523, 11620,    12,
            2,  1429,    65,    43,  5344,  3522,  3902])
tensor([[   1,   21,  815, 2711, 9184,   42,  812,   65,  117,  319, 1399]]) tensor([ 815, 2711, 1951,   43, 4928, 1531,   65, 2530, 2490,   12, 1271, 2711,
        2043,   43, 1399,   65,    4,  117, 4886,  104, 6748,   43, 3198,   65,
        1975, 1663,   27, 1975, 9153,  717,   27,   43, 3069,  423,   65, 5638,
        4497])
tensor([[  1, 811,  21,  43, 117, 242, 326]]) tensor([  43,  117,  242,  277,    4, 1235,  403,   65,  298,   12,  108,   33,
          80, 3123,  485,  132,   27, 1016,   12,  102,   59,  265])
tensor([[ 225,   54,  132,  232,  233,  340,  343,  117, 1241,   12,  678,   89]]) tensor([ 132,   54,  232,  233,  340,  343,  117, 1241,   12,  678,   89

In [42]:
import torch.nn as nn

In [43]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [ ]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 3])
shape of b: torch.Size([1, 3, 50])
shape of c: torch.Size([1, 3, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [54]:
learning_rate = 0.001
epochs = 20

In [55]:
model = SimpleRNN(len(vocab))

In [56]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [64]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[4])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

IndexError: index 4 is out of bounds for dimension 0 with size 1

In [58]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [59]:
predict(model, "What is the largest planet in our solar system?")

I don't know
or


In [60]:
list(vocab.keys())[7]

'altimesh'